# Fase 3 · Núcleo algorítmico
**Airbnb en Santiago: Grupo 4 · MCDI500**

## 1. Configuración
Se declaran las variables y rutas utilizadas para verificar las transformaciones de F2.

In [1]:
# =====================================================================
# CONFIGURACIÓN DEL PROYECTO
# =====================================================================

# Ruta relativa a la raíz del proyecto
RUTA_DATOS = "data/processed/listings_procesado.csv"

COLUMNA_PRECIO = "price_clp"       # precio en CLP por noche
COLUMNA_ID = "id"                    # identificador: se elimina del análisis

COLUMNAS_CONTINUAS = [
    # ubicación
    "latitude", "longitude",
    # tamaño y capacidad
    "accommodates", "bedrooms", "beds", "banos_numero", "amenities_count",
    # reglas de estadía
    "minimum_nights", "maximum_nights",
    # disponibilidad y demanda
    "availability_30", "availability_60", "availability_90", "availability_365",
    "number_of_reviews", "number_of_reviews_ltm", "number_of_reviews_l30d",
    "reviews_per_month", "estimated_occupancy_l365d", "antiguedad_dias",
    # puntajes de reseñas
    "review_scores_rating", "review_scores_accuracy", "review_scores_cleanliness",
    "review_scores_checkin", "review_scores_communication",
    "review_scores_location", "review_scores_value",
    # anfitrión
    "calculated_host_listings_count",
]

COLUMNAS_NOMINALES = ["neighbourhood_cleansed", "property_type", "propiedad_agrupada",
                      "host_is_superhost"]   # superhost viene como texto t/f

COLUMNAS_BINARIAS = [
    "bedrooms_imputado", "beds_imputado", "ocupacion_tope", "es_multianfitrion",
    "amen_kitchen", "amen_wifi", "amen_hot_water", "amen_hair_dryer",
    "amen_cooking_basics", "amen_dishes_and_silverware", "amen_hangers",
    "amen_bed_linens", "amen_tv", "amen_microwave", "amen_self_check_in",
    "amen_refrigerator", "amen_iron", "amen_hot_water_kettle", "amen_wine_glasses",
    "amen_elevator", "amen_extra_pillows_and_blankets", "amen_long_term_stays_allowed",
    "amen_oven", "amen_toaster",
]

# Columnas del archivo que se dejan fuera a propósito (ver notas abajo)
COLUMNAS_EXCLUIDAS = ["host_id", "room_type", "banos_compartidos", "first_review",
                      "last_review", "precio_por_huesped", "estimated_revenue_l365d"]

SEMILLA = 42


# Datos de demostración: solo para revisar el cuaderno sin el archivo real.
# Con True, los resultados no son del proyecto. En la entrega debe ir False.
PERMITIR_DEMOSTRACION = False
# =====================================================================

COLUMNAS_ESPERADAS = ([COLUMNA_ID, COLUMNA_PRECIO] + COLUMNAS_CONTINUAS
                      + COLUMNAS_NOMINALES + COLUMNAS_BINARIAS)

from pathlib import Path


def localizar_raiz():
    """Devuelve la raiz relativa segun la carpeta de trabajo de Jupyter."""
    return Path("..") if Path.cwd().name in {"F1", "F2", "F3", "F4"} else Path(".")


RAIZ = localizar_raiz()
RUTA_DATOS = str(RAIZ / RUTA_DATOS)
DIR_PROCESADO = str(RAIZ / "data" / "processed")   # salidas compartidas por las fases

print("Raiz del proyecto   :", Path.cwd().parent.name if RAIZ == Path("..") else Path.cwd().name)
print("Archivo configurado :", Path(RUTA_DATOS))
print("Variable de precio  :", COLUMNA_PRECIO)
print("Columnas esperadas  :", len(COLUMNAS_ESPERADAS))

Raiz del proyecto   : proyecto-grupo4-mcdi500
Archivo configurado : ../data/processed/listings_procesado.csv
Variable de precio  : price_clp
Columnas esperadas  : 57


## 2. Entorno
Se registran las versiones del entorno y la semilla de ejecución.

In [2]:
import os
import sys
import time
import timeit
import tracemalloc

import numpy as np
import pandas as pd

SEMILLA = 42
np.random.seed(SEMILLA)

_p = Path(sys.executable).parts
print("Python :", Path(*_p[_p.index(".venv"):]) if ".venv" in _p else sys.executable)
print("pandas :", pd.__version__)
print("NumPy  :", np.__version__)
print("Semilla:", SEMILLA)

Python : .venv/bin/python
pandas : 3.1.1
NumPy  : 2.5.4
Semilla: 42


## 3. Carga
El procesado de F2 se carga con comprobación de columnas y conversión de tipos numéricos.

In [3]:
# Las funciones de carga se movieron a src/carga.py para reutilizarlas
# desde cualquier fase. El cuaderno las importa en vez de definirlas.
DIR_SRC = RAIZ / "src"

from src.carga import cargar

datos = cargar(RUTA_DATOS, COLUMNAS_ESPERADAS, COLUMNAS_CONTINUAS)
datos.head()


Forma: 3313 filas x 64 columnas
Esquema verificado: 57 columnas declaradas, 7 no declaradas
  No declaradas: ['host_id', 'room_type', 'banos_compartidos', 'precio_por_huesped', 'estimated_revenue_l365d', 'first_review', 'last_review']


,id,host_id,neighbourhood_cleansed,latitude,longitude,property_type,propiedad_agrupada,room_type,accommodates,bedrooms,...,review_scores_rating,review_scores_accuracy,review_scores_cleanliness,review_scores_checkin,review_scores_communication,review_scores_location,review_scores_value,host_is_superhost,calculated_host_listings_count,es_multianfitrion
0,1226526,828029c6c29f,Providencia,-33.42681,-70.60961,Entire rental unit,Entire rental unit,Entire home/apt,4,2.0,...,4.67,4.74,4.83,4.91,4.92,4.94,4.72,t,2,1
1,1300359,250fdbee5201,Providencia,-33.43005,-70.61772,Entire rental unit,Entire rental unit,Entire home/apt,4,2.0,...,4.73,4.71,4.80,4.79,4.85,4.85,4.61,f,20,1
2,3735992,481459247d1a,Estación Central,-33.45593,-70.70020,Entire condo,Entire condo,Entire home/apt,5,2.0,...,4.71,4.81,4.69,4.76,4.90,4.28,4.62,t,9,1
3,3810101,9e622c2063be,Lo Barnechea,-33.35603,-70.24830,Entire rental unit,Entire rental unit,Entire home/apt,5,2.0,...,4.91,4.89,4.71,4.89,4.86,4.89,4.77,f,1,0
4,1446357,476b08a4a2a1,Lo Barnechea,-33.34869,-70.29280,Entire condo,Entire condo,Entire home/apt,6,2.0,...,4.68,4.64,4.51,4.67,4.89,4.83,4.42,f,1,0


El perfil muestra los tipos y valores faltantes del procesado.

In [4]:
def perfilar(df):
    """Devuelve una fila por columna con su tipo, únicos y porcentaje de nulos."""
    return pd.DataFrame({
        "columna": df.columns,
        "tipo": [str(t) for t in df.dtypes],
        "unicos": [df[c].nunique(dropna=True) for c in df.columns],
        "nulos": df.isna().sum().values,
        "pct_nulos": (df.isna().mean() * 100).round(2).values,
    }).sort_values("pct_nulos", ascending=False).reset_index(drop=True)


perfil = perfilar(datos)
faltantes = perfil[perfil["nulos"] > 0]
print("Columnas con valores faltantes:")
if faltantes.empty:
    print("  Ninguna: el conjunto de datos no tiene valores faltantes.")
else:
    print(faltantes.to_string(index=False))

# La variable objetivo es continua: se describe con estadísticos, no con frecuencias
print(f"\nDistribución del precio '{COLUMNA_PRECIO}':")
print(datos[COLUMNA_PRECIO].describe().round(2))
print(f"Asimetría (skew)          : {datos[COLUMNA_PRECIO].skew():.2f}")
print(f"Asimetría tras log1p      : {np.log1p(datos[COLUMNA_PRECIO]).skew():.2f}")

print("\nCategorías de property_type (ordenadas por frecuencia):")
print(datos["property_type"].value_counts())

print("\nCategorías de propiedad_agrupada:")
print(datos["propiedad_agrupada"].value_counts())

Columnas con valores faltantes:
  Ninguna: el conjunto de datos no tiene valores faltantes.

Distribución del precio 'price_clp':
count        3313.00
mean        98274.27
std        650820.17
min         19125.00
25%         44760.00
50%         59950.00
75%         85500.00
max      37034021.50
Name: price_clp, dtype: float64
Asimetría (skew)          : 55.25
Asimetría tras log1p      : 1.83

Categorías de property_type (ordenadas por frecuencia):
property_type
Entire rental unit           2926
Entire condo                  168
Entire serviced apartment      76
Entire home                    66
Entire loft                    30
Entire vacation home            9
Entire cabin                    8
Entire guesthouse               8
Tiny home                       7
Entire guest suite              4
Entire chalet                   3
Entire villa                    2
Camper/RV                       1
Entire cottage                  1
Casa particular                 1
Windmill              

## 4. Transformaciones con estado
Las clases representan los pasos que aprenden parámetros: medianas de camas
y dormitorios por tramo de capacidad, categorías frecuentes de propiedad y
veinte comodidades frecuentes. `ajustar` aprende y `transformar` aplica.
La propiedad `parametros` devuelve una copia profunda del estado aprendido.

In [5]:
from src.transformadores import (ImputadorPorGrupo, AgrupadorTopCategorias,
                          BinarizadorAmenidades, parsear_amenidades)
from src.pipeline import Pipeline
from src import utilidades
from src.medicion import medir

RUTA_MUESTRA = str(RAIZ / "data" / "raw" / "listings.csv")
RUTA_PROCESADO = str(RAIZ / "data" / "processed" / "listings_procesado.csv")
DIR_DOCS = RAIZ / "docs"

print("Módulos importados desde src/:")
for modulo in ("carga", "transformadores", "pipeline", "medicion", "utilidades"):
    print(f"  {modulo}: {DIR_SRC / (modulo + '.py')}")


Módulos importados desde src/:
  carga: ../src/carga.py
  transformadores: ../src/transformadores.py
  pipeline: ../src/pipeline.py
  medicion: ../src/medicion.py
  utilidades: ../src/utilidades.py


### Entrada comparable con F2
Se parte de la muestra original, se convierte precio y se aplica el mismo
filtro de alojamientos completos activos. Las filas sin precio se excluyen,
igual que en F2. La comparación se limita a las columnas de los transformadores.

In [6]:
from src.carga import leer_archivo

# La muestra cruda tiene las 78 columnas: se carga sin verificar esquema
df_pre = leer_archivo(RUTA_MUESTRA)
print(f"Muestra cruda: {df_pre.shape}")

# price llega como texto: la conversion es de F2 y se repite aqui
df_pre["price_clp"] = (df_pre["price"]
                       .str.replace("$", "", regex=False)
                       .str.replace(",", "", regex=False)
                       .astype(float))

# Universo (D1, D2) y fuera las filas sin precio
df_pre = df_pre[(df_pre["room_type"] == "Entire home/apt")
                & (df_pre["number_of_reviews_ltm"] > 0)
                & (df_pre["availability_365"] > 0)].copy()
n_antes = len(df_pre)
df_pre = df_pre.dropna(subset=["price_clp"])
print(f"Universo: {n_antes} -> {len(df_pre)} tras quitar filas sin precio")
print("Esto debe coincidir con la entrada que usó F2 para aprender.")


Muestra cruda: (5106, 78)
Universo: 3343 -> 3313 tras quitar filas sin precio
Esto debe coincidir con la entrada que usó F2 para aprender.


In [7]:
# El pipeline con clases: cada paso aprende su parámetro en ajustar()
# y lo aplica en transformar(). El orden es el mismo que usó F2.
pasos = [
    ImputadorPorGrupo("beds", "accommodates"),
    ImputadorPorGrupo("bedrooms", "accommodates"),
    AgrupadorTopCategorias("property_type", n_top=5,
                          nombre_salida="propiedad_agrupada"),
    BinarizadorAmenidades("amenities", n_top=20),
]
pipeline = Pipeline(pasos)
pipeline.ajustar(df_pre)
resultado = pipeline.transformar(df_pre)

print(pipeline.resumen().to_string(index=False))
print(f"\nResultado: {resultado.shape}")


 orden                                  paso                  clase
     1               ImputadorPorGrupo(beds)      ImputadorPorGrupo
     2           ImputadorPorGrupo(bedrooms)      ImputadorPorGrupo
     3 AgrupadorTopCategorias(property_type) AgrupadorTopCategorias
     4      BinarizadorAmenidades(amenities)  BinarizadorAmenidades

Resultado: (3313, 103)


## 5. Verificación frente a F2
Se comprueba el orden de las filas por `id` y la igualdad de las 26 columnas
producidas por las clases. El resultado intermedio conserva otras columnas
originales: no se presenta como una reproducción completa del procesado final de F2.

In [8]:
procesado = leer_archivo(RUTA_PROCESADO)

# Verifica la entrada y las columnas que producen las clases.
assert df_pre["id"].is_unique, "La entrada tiene identificadores repetidos"
assert np.isfinite(df_pre[COLUMNA_PRECIO]).all(), "El precio tiene valores no finitos"
assert (df_pre[COLUMNA_PRECIO] > 0).all(), "El precio debe ser positivo"
assert resultado.columns.is_unique, "El resultado tiene columnas repetidas"
assert resultado["id"].is_unique, "El resultado tiene identificadores repetidos"
assert not resultado[["beds", "bedrooms", "amenities_count"]].isna().any().any(), \
    "Quedaron valores vacios en columnas transformadas"

cols_amen = [c for c in procesado.columns if c.startswith("amen_")]
cols_amen_resultado = [c for c in resultado.columns if c.startswith("amen_")]
assert set(cols_amen_resultado) == set(cols_amen), \
    "Las columnas de amenidades no coinciden con F2"

columnas_binarias = ["beds_imputado", "bedrooms_imputado"] + cols_amen_resultado
assert resultado[columnas_binarias].isin([0, 1]).all().all(), \
    "Hay valores distintos de 0/1 en columnas binarias"

# Las filas deben coincidir con F2, en el mismo orden.
assert resultado["id"].tolist() == procesado["id"].tolist(), \
    "Las filas no coinciden con el procesado de F2"
print(f"[OK] {len(resultado)} filas en el mismo orden que F2")

# Compara las columnas creadas por las clases con el resultado de F2.
columnas_clase = (["beds", "beds_imputado", "bedrooms", "bedrooms_imputado",
                   "propiedad_agrupada", "amenities_count"] + cols_amen)
faltantes = [col for col in columnas_clase if col not in resultado.columns]
assert not faltantes, f"Faltan columnas transformadas: {faltantes}"

fallas = []
for col in columnas_clase:
    try:
        pd.testing.assert_series_equal(
            resultado[col].reset_index(drop=True),
            procesado[col].reset_index(drop=True),
            check_names=False)
    except AssertionError as error:
        fallas.append((col, str(error).splitlines()[0]))

if fallas:
    raise AssertionError(f"Diferencias con F2: {fallas}")
print(f"[OK] {len(columnas_clase)} columnas identicas a F2 "
      f"({len(cols_amen)} de amenidades incluidas)")


[OK] 3313 filas en el mismo orden que F2
[OK] 26 columnas identicas a F2 (20 de amenidades incluidas)


## Medición de eficiencia

Se comparan `map` y `explode` con `pivot_table` para 500, 1.500 y todas las
filas. Antes de medir se comprueba que ambas versiones entreguen el mismo
resultado. Se conserva el mínimo de siete repeticiones y se mide memoria
por separado en cada tamaño. `tracemalloc` registra asignaciones de Python;
no representa necesariamente toda la memoria nativa de pandas y NumPy.

In [9]:
import statistics

listas_completas = df_pre["amenities"].map(parsear_amenidades)
top_amen = pipeline.pasos[-1].parametros["top"]

def binarias_bucle(lista_items):
    salida = {}
    for amenidad in top_amen:
        salida["amen_" + utilidades.normalizar_nombre(amenidad)] = \
            lista_items.map(lambda lista, item=amenidad: int(item in lista))
    return pd.DataFrame(salida, index=lista_items.index)

def binarias_vectorizada(lista_items):
    largas = lista_items.explode().rename("amenidad").to_frame()
    largas["tiene"] = 1
    matriz = largas.pivot_table(index=largas.index, columns="amenidad",
                                values="tiene", aggfunc="max", fill_value=0)
    matriz = matriz.reindex(columns=top_amen, fill_value=0)
    matriz.columns = ["amen_" + utilidades.normalizar_nombre(c)
                      for c in matriz.columns]
    return matriz.astype(int)

A = binarias_bucle(listas_completas)
B = binarias_vectorizada(listas_completas)
pd.testing.assert_frame_equal(A.reset_index(drop=True), B.reset_index(drop=True))
print(f"[OK] Ambas formas entregan las mismas {len(top_amen)} columnas")

rep = 7
tamanos = sorted(set([500, 1500, len(df_pre)]))
filas_medicion = []
for n in tamanos:
    lista_items = listas_completas.head(n)

    def bucle():
        return binarias_bucle(lista_items)

    def vectorizada():
        return binarias_vectorizada(lista_items)

    pd.testing.assert_frame_equal(bucle(), vectorizada())
    tiempos_bucle = timeit.repeat(bucle, number=1, repeat=rep)
    tiempos_vectorizada = timeit.repeat(vectorizada, number=1, repeat=rep)
    t_bucle = min(tiempos_bucle)
    t_vectorizada = min(tiempos_vectorizada)

    if t_bucle <= t_vectorizada:
        ganador = "map"
        razon = t_vectorizada / t_bucle
    else:
        ganador = "explode()"
        razon = t_bucle / t_vectorizada

    _, _, memoria_map = medir(bucle)
    _, _, memoria_explode = medir(vectorizada)
    filas_medicion.append({
        "filas": n,
        "map_segundos": t_bucle,
        "map_mb": memoria_map,
        "explode_mb": memoria_explode,
        "explode_segundos": t_vectorizada,
        "ganador": ganador,
        "veces_mas_rapido": razon,
    })

tabla_medicion = pd.DataFrame(filas_medicion)
print(tabla_medicion.to_string(index=False, formatters={
    "map_segundos": lambda v: f"{v:.4f}",
    "explode_segundos": lambda v: f"{v:.4f}",
    "veces_mas_rapido": lambda v: f"{v:.1f}",
}))

_, _, mb_bucle = medir(lambda: binarias_bucle(listas_completas))
_, _, mb_vectorizada = medir(lambda: binarias_vectorizada(listas_completas))
print(f"Memoria pico Python ({len(df_pre)} filas): map {mb_bucle:.2f} MB | explode() {mb_vectorizada:.2f} MB")


tabla_medicion.to_csv(DIR_DOCS / "eficiencia_f3.csv", index=False)

[OK] Ambas formas entregan las mismas 20 columnas


 filas map_segundos   map_mb  explode_mb explode_segundos ganador veces_mas_rapido
   500       0.0055 0.174740    4.176661           0.0114     map              2.1
  1500       0.0156 0.479916   19.647783           0.0262     map              1.7
  3313       0.0359 1.033199   54.137182           0.0494     map              1.4
Memoria pico Python (3313 filas): map 1.03 MB | explode() 54.14 MB


**Interpretación.** La comparación y sus razones quedan en la tabla anterior
y en `docs/eficiencia_f3.csv`. Los tiempos dependen del equipo; el informe
debe tomar las cifras de esa ejecución, no de una anterior.

Con `n` alojamientos, `m` comodidades seleccionadas y `a` elementos medios
por lista, `map` requiere O(n·m·a) tiempo por la búsqueda en listas y
O(n·m) memoria de salida. `explode` crea O(n·a) registros; el pivote agrega
costos de agrupación y una matriz de hasta O(n·u), donde `u` es la cantidad
de comodidades distintas antes de seleccionar las `m` finales. Ambas
versiones generan una salida O(n·m). No se deduce complejidad solo de tres
mediciones ni se supone que vectorizar siempre reduzca el costo.

## 7. Recursión
`aplanar` recorre los metadatos anidados hasta encontrar valores simples y
los convierte en pares clave–valor. La profundidad puede variar. La recursión
es adecuada para esta estructura pequeña; una pila iterativa evitaría el
límite de llamadas si se aplicara a estructuras profundas.

In [10]:
from src.utilidades import aplanar

# Metadatos de la Fase 3 con la misma estructura anidada que uso F2
METADATOS_F3 = {
    "proyecto": {"nombre": "Alojamientos Airbnb en Santiago",
                 "grupo": "Grupo 4", "fase": "F3", "semilla": SEMILLA},
    "nucleo": {"clase_base": "Transformador",
               "clases": ["ImputadorPorGrupo", "AgrupadorTopCategorias",
                          "BinarizadorAmenidades"],
               "pipeline": "Pipeline (polimorfismo)"},
    "verificacion": {"contra": "data/processed/listings_procesado.csv",
                     "columnas_comparadas": len(columnas_clase)},
    "medicion": {"comparacion": "bucle vs explode()",
                 "filas": len(df_pre), "amenidades": len(top_amen),
                 "repeticiones": rep, "estadistico": "minimo",
                 "tabla": "docs/eficiencia_f3.csv",
                 "ganador": tabla_medicion.iloc[-1]["ganador"]},
}

metadatos_planos = aplanar(METADATOS_F3)
for clave, valor in metadatos_planos.items():
    print(f"  {clave:<40} {valor}")

# Guarda la evidencia de la fase en formato de tabla.
pd.DataFrame(list(metadatos_planos.items()),
             columns=["clave", "valor"]).to_csv(
    DIR_DOCS / "metadatos_fase3.csv", index=False)
print(f"\nGuardado en docs/metadatos_fase3.csv")


  proyecto.nombre                          Alojamientos Airbnb en Santiago
  proyecto.grupo                           Grupo 4
  proyecto.fase                            F3
  proyecto.semilla                         42
  nucleo.clase_base                        Transformador
  nucleo.clases                            ImputadorPorGrupo, AgrupadorTopCategorias, BinarizadorAmenidades
  nucleo.pipeline                          Pipeline (polimorfismo)
  verificacion.contra                      data/processed/listings_procesado.csv
  verificacion.columnas_comparadas         26
  medicion.comparacion                     bucle vs explode()
  medicion.filas                           3313
  medicion.amenidades                      20
  medicion.repeticiones                    7
  medicion.estadistico                     minimo
  medicion.tabla                           docs/eficiencia_f3.csv
  medicion.ganador                         map

Guardado en docs/metadatos_fase3.csv


## 8. Arquitectura y patrón de diseño
`Transformador` aplica **Template Method**: fija la secuencia de ajuste y
transformación, mientras las subclases implementan `aprender` y `aplicar`.
La base abstracta evita crear un transformador incompleto. El estado interno
se consulta con una propiedad que devuelve copia.

`Pipeline` compone los pasos y los recorre mediante la misma interfaz. El
polimorfismo permite ejecutar imputación, agrupación y binarización sin
preguntar por su clase concreta. No se presentan como estrategias alternativas.

| Módulo | Responsabilidad |
|---|---|
| `carga.py` | Lectura y verificación de columnas |
| `transformadores.py` | Aprendizaje y aplicación de transformaciones |
| `pipeline.py` | Secuencia de pasos |
| `medicion.py` | Tiempo y memoria |
| `utilidades.py` | Normalización y metadatos |

```text
Transformador (ABC)
 ├── ImputadorPorGrupo
 ├── AgrupadorTopCategorias
 └── BinarizadorAmenidades
Pipeline ── contiene ──> Transformador [0..n]
```

Cada transformador concentra una operación; el pipeline solo conoce su
contrato. Cambiar la imputación requiere modificar su implementación, no
las clases de agrupación o binarización. La igualdad con F2 verifica que la
separación mantuvo las transformaciones comparadas.

## 9. Validación
El caso normal se contrasta con F2. Los siguientes ejemplos verifican entradas
sin faltantes y errores esperados. Las pruebas en `tests/` incluyen además
una fila, capacidad constante, categorías nuevas, listas vacías y copia de parámetros.

In [11]:
def comprobar_error(tipo, funcion, etiqueta):
    try:
        funcion()
    except tipo as error:
        print(f"[OK] {etiqueta}: {error}")
    else:
        raise AssertionError(f"{etiqueta}: no se produjo {tipo.__name__}")

# Caso limite 1: columna sin nulos
# La bandera queda en 0 y los valores no se alteran.
df_sin_nulos = df_pre.copy()
df_sin_nulos["beds"] = df_sin_nulos["beds"].fillna(2)
imp = ImputadorPorGrupo("beds", "accommodates").ajustar_transformar(df_sin_nulos)
assert (imp["beds_imputado"] == 0).all(), "No debe haber imputados sin nulos"
assert imp["beds"].equals(df_sin_nulos["beds"]), "No debe alterar valores"
print("[OK] Limite: columna sin nulos no se altera")

# Caso limite 2: una sola categoria en property_type
df_una_cat = df_pre.copy()
df_una_cat["property_type"] = "Entire rental unit"
agr = AgrupadorTopCategorias("property_type", n_top=5,
                             nombre_salida="propiedad_agrupada")
res_una = agr.ajustar_transformar(df_una_cat)
assert (res_una["propiedad_agrupada"] == "Entire rental unit").all(), \
    "Con una sola categoria no debe aparecer 'Otro'"
print("[OK] Limite: una sola categoria no genera 'Otro'")

# Caso limite 3: amenities vacias y categoria no vista al ajustar.
df_vacias = df_pre.copy()
df_vacias["amenities"] = "[]"
bin_v = BinarizadorAmenidades("amenities", n_top=20).ajustar_transformar(df_vacias)
assert (bin_v["amenities_count"] == 0).all(), "El conteo debe ser 0"
print("[OK] Limite: amenities vacias dan conteo 0")

agr_ajustado = AgrupadorTopCategorias("property_type").ajustar(
    pd.DataFrame({"property_type": ["Entire condo"]}))
nueva_categoria = agr_ajustado.transformar(
    pd.DataFrame({"property_type": ["Categoria nueva"]}))
assert nueva_categoria.loc[0, "property_type_agrupada"] == "Otro"
print("[OK] Limite: categoria no vista se agrupa como 'Otro'")

# Las excepciones esperadas deben ocurrir; si no, la prueba falla.
comprobar_error(
    RuntimeError,
    lambda: ImputadorPorGrupo("beds", "accommodates").transformar(df_pre),
    "Transformar antes de ajustar")
comprobar_error(
    KeyError,
    lambda: ImputadorPorGrupo("columna_que_no_existe", "accommodates").ajustar(df_pre),
    "Columna inexistente")
comprobar_error(
    TypeError,
    lambda: Pipeline().agregar("no soy un transformador"),
    "Tipo de paso incorrecto")
comprobar_error(
    ValueError,
    lambda: parsear_amenidades('["Wifi"'),
    "JSON de amenities mal formado")

df_sin_datos = pd.DataFrame({"accommodates": [1, 2, 3], "beds": [None, None, None]})
comprobar_error(
    ValueError,
    lambda: ImputadorPorGrupo("beds", "accommodates").ajustar(df_sin_datos),
    "Columna de imputacion completamente vacia")


[OK] Limite: columna sin nulos no se altera
[OK] Limite: una sola categoria no genera 'Otro'
[OK] Limite: amenities vacias dan conteo 0
[OK] Limite: categoria no vista se agrupa como 'Otro'
[OK] Transformar antes de ajustar: ImputadorPorGrupo(beds): hay que ajustar antes de transformar.
[OK] Columna inexistente: 'ImputadorPorGrupo(columna_que_no_existe): la columna no existe en el conjunto.'
[OK] Tipo de paso incorrecto: Se esperaba un Transformador y se recibio str.
[OK] JSON de amenities mal formado: amenities debe contener una lista JSON valida.
[OK] Columna de imputacion completamente vacia: ImputadorPorGrupo(beds): no hay valores disponibles para calcular la mediana.


## 10. Cierre
Las transformaciones con estado quedan en clases y se verifican contra las
columnas correspondientes de F2. La tabla de eficiencia registra tiempo y
memoria para tres tamaños. F4 utiliza el procesado de F2 y este núcleo como
evidencia de equivalencia de los pasos aprendidos.

La arquitectura sigue el material de Semana 2 (Universidad Andrés Bello, 2026a); el registro de
eficiencia usa `timeit` y `tracemalloc` de Python. La limitación principal es
que la comparación no cubre las operaciones sin estado que permanecen en F2.

## Referencia
Universidad Andrés Bello. (2026a). *Material docente de la Fase 3, Semana 2* [Material de curso]. MCDI500.